In [ ]:
# ==========================================
# 0. CLEAR WORKSPACE (OPTIONAL)
# ==========================================

try:
    from IPython import get_ipython
    get_ipython().run_line_magic('reset', '-f')
except:
    pass


# ==========================================
# 1. IMPORTS
# ==========================================

# --- Standard Library ---
import os
from datetime import datetime

# --- Data Manipulation & Math ---
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde

# --- Visualization ---
import plotly.graph_objects as go

In [ ]:
# ===================================================================
# 0. CONFIGURATION
# ===================================================================

# Set base path and data vintage
base_path = "/path/to/ITNOW/"
vintage = "20260509"

# Filter dates for estimation and evaluation samples
Est_End = "2006-04-01"  # Do not change this date
Eval_End = "2026-04-01"

# Create current vintage figure folder
fig_dir = os.path.join(base_path, "HISTORY/FIGURES", f"json_{vintage}")
os.makedirs(fig_dir, exist_ok=True)

# Chart size for consistency
width0 = 800
height0 = 500

In [ ]:
# ===================================================================
# 1. INITIALIZATION: DATA LOADING & PREPARATION
# ===================================================================

print("--- Initializing Data ---")

# --- Load historical GDP and prediction data ---
try:
    # Paths
    path_predicted = os.path.join(base_path, "HISTORY/PREDICTION/", f"{vintage}_fcasts_covid.csv")

    # Load .csv file
    y_predicted = pd.read_csv(path_predicted, )

    # Convert 'Time' column to datetime and set as index
    y_predicted['Time'] = pd.to_datetime(y_predicted['Time'])
    y_predicted.set_index('Time', inplace=True)
        
    # Load actual GDP history
    path_actual = os.path.join(base_path, "DATA/GDP_History/", f"GDP{vintage}.csv")
    y_actual = pd.read_csv(path_actual)
    y_actual['Time'] = pd.to_datetime(y_actual['Time'])
    y_actual.set_index('Time', inplace=True)
    
    print("Files loaded successfully.")
except FileNotFoundError as e:
    print(f"Error loading files: {e}. Please check paths and vintage.")
    exit()

In [ ]:
# --- Filter data to the correct evaluation sample ---
y_actual = y_actual.loc[Est_End:Eval_End]

print(f"Data filtered from {Est_End} to {Eval_End}.")

# --- Compute LEVEL predictions ---
# Start with the actual level data
y_level = y_actual[['Level']].copy()

# Find the first valid nowcast to align calculations
first_fcast_idx = y_predicted['BG_K1_4'].first_valid_index()
first_fcast_loc = y_predicted.index.get_loc(first_fcast_idx)

# Calculate the nowcasted level: Level(t) = Level(t-3) * (1 + Growth_Nowcast(t))
# We use .shift(3) because growth is QoQ, and data is monthly (3 months = 1 quarter)
level_nowcast_series = y_actual['Level'].shift(3) * (1 + y_predicted['BG_K1_4'])

# Create the new column, initially all NaN
y_level['BG_K1_4'] = np.nan
# Fill in the calculated values starting from the first available nowcast
y_level.iloc[first_fcast_loc:, y_level.columns.get_loc('BG_K1_4')] = level_nowcast_series.iloc[first_fcast_loc:]

# Convert levels from millions to thousands of millions (billions)
y_level['Level'] = y_level['Level'] / 1000
y_level['BG_K1_4'] = y_level['BG_K1_4'] / 1000

print("Level nowcasts calculated.")

In [ ]:
# --- Compute GROWTH predictions and prediction intervals ---
# Start with actuals and add prediction columns
y_growth = y_actual.copy()
a = 0.05  # For 90% prediction intervals (1 - 2*a)

# Process BG_K1_4 model
y_growth['BG_K1_4'] = y_predicted['BG_K1_4']
fcast_errors_bg_1 = y_growth['Growth'] - y_growth['BG_K1_4']
y_growth['BG_K1_4_Low'] = y_growth['BG_K1_4'] + fcast_errors_bg_1.quantile(a)
y_growth['BG_K1_4_Upp'] = y_growth['BG_K1_4'] + fcast_errors_bg_1.quantile(1 - a)

print("Growth prediction intervals calculated.\n")

In [ ]:
# ===================================================================
# 2. FIGURE: Actual Nominal GDP Growth vs Nowcast
# ===================================================================

print("--- Generating Figure 1: GDP Growth vs Nowcast ---")

# Create the "Legend" items dynamically
last_date = y_growth.index[-1]
last_nowcast_val = y_growth['BG_K1_4'].iloc[-1]
LegendItemFig1 = [
    "Quarterly Nominal GDP Growth",
    "Historical Nowcast",
    f"{last_date.strftime('%b')}-{last_date.year} Nowcast ({last_nowcast_val:.3%})",
    f"{(1 - 2*a):.0%} Prediction Interval"
]

# Prepare in-between coordinates for the prediction interval, starting from the first valid nowcast
valid_growth_data = y_growth.loc[first_fcast_idx:].copy()
inBetweenCoord = np.concatenate([valid_growth_data.index.values, valid_growth_data.index.values[::-1]])
inBetween_DT_1 = pd.concat([valid_growth_data['BG_K1_4_Low'], valid_growth_data['BG_K1_4_Upp'][::-1]]).values

# Plot with Plotly
fig1 = go.Figure()

# Add fill for the prediction interval
fig1.add_trace(go.Scatter(
    x=inBetweenCoord, 
    y=inBetween_DT_1, 
    fill='toself',
    fillcolor='rgba(130, 130, 130, 0.5)',
    line=dict(color='rgba(255, 255, 255, 0)'),
    name=LegendItemFig1[3],
    hovertemplate='%{x|%b %Y}<br>PI Bound: %{y:.3%}<extra></extra>'
))

# Add horizontal line at y=0
fig1.add_hline(y=0, line_dash="dot", line_color="black", line_width=1)

# Add scatter plot for actual GDP Growth
fig1.add_trace(go.Scatter(
    x=y_growth.index, 
    y=y_growth['Growth'], 
    mode='markers',
    name=LegendItemFig1[0],
    marker=dict(size=12, color='#0072BD'),
    hovertemplate='%{x|%b %Y}<br>GDP Growth: %{y:.3%}<extra></extra>'
))

# Add line plot for the historical nowcast
fig1.add_trace(go.Scatter(
    x=y_growth.index, 
    y=y_growth['BG_K1_4'], 
    mode='lines',
    name=LegendItemFig1[1],
    line=dict(color='#D95319', width=3),
    hovertemplate='%{x|%b %Y}<br>Nowcast: %{y:.3%}<extra></extra>'
))

# Add a prominent point for the latest Nowcast value
fig1.add_trace(go.Scatter(
    x=[last_date], 
    y=[last_nowcast_val], 
    mode='markers',
    marker=dict(size=20, color='#EDB120', line=dict(width=2, color='DarkSlateGrey')),
    name=LegendItemFig1[2],
    hovertemplate='%{x|%b %Y}<br>Latest Nowcast: %{y:.3%}<extra></extra>'
))

# Update layout for the figure
fig1.update_layout(
    yaxis_title="Quarterly Nominal GDP Growth",
    yaxis_tickformat=".1%",
    xaxis=dict(tickformat='%Y', tickangle=45, dtick="M12"),
    width=width0,
    height=height0,
    legend=dict(
        orientation="h",
        yanchor="bottom", y=1.02,
        xanchor="center", x=0.5
    ),
    font=dict(family="Times New Roman", size=18)
)

fig1.show()
fig1_path = os.path.join(base_path, "HISTORY/FIGURES/", f"json_{vintage}", "GDP_Growth_Ncast.json")
fig1.write_json(fig1_path)

In [ ]:
# ===================================================================
# 3. FIGURE: Actual Nominal GDP Level vs Nowcast
# ===================================================================

print("--- Generating Figure 2: GDP Level vs Nowcast ---")

# Prepare legend items
LegendItemFig2 = [
    "Quarterly GDP (2020 Chain Linked)",
    "Historical Nowcast",
    f"Q{last_date.quarter}-{last_date.year} Nowcast"
]

# Plot with Plotly
fig2 = go.Figure()

# Add bar chart for actual GDP Level
fig2.add_trace(go.Bar(
    x=y_level.index, 
    y=y_level['Level'], 
    name=LegendItemFig2[0],
    marker=dict(color='#0072BD'),
    hovertemplate='%{x|%b %Y}<br>Level: %{y:.3f} Bln €<extra></extra>'
))

# Add line plot for the nowcasted level (plot every 3rd point for clarity, like MATLAB)
y_level_quarterly_plot = y_level.iloc[first_fcast_loc::3]
fig2.add_trace(go.Scatter(
    x=y_level_quarterly_plot.index, 
    y=y_level_quarterly_plot['BG_K1_4'], 
    mode='lines', 
    name=LegendItemFig2[1],
    line=dict(color='#D95319', width=4),
    hovertemplate='%{x|%b %Y}<br>Nowcast: %{y:.3f} Bln €<extra></extra>'
))

# Add horizontal line and annotation for the latest nowcast
last_level_nowcast = y_level['BG_K1_4'].iloc[-1]
fig2.add_hline(y=last_level_nowcast, line_dash="dot", line_color="#D95319")
fig2.add_annotation(
    x=y_level.index[-25],
    y=last_level_nowcast + 10, 
    text=f"{last_level_nowcast:.3f} billions of €",
    showarrow=False,
    font=dict(family="Times New Roman", size=20),
)

# Add a prominent point for the final nowcast
fig2.add_trace(go.Scatter(
    x=[last_date], 
    y=[last_level_nowcast], 
    mode='markers',
    marker=dict(size=15, color='#EDB120', line=dict(width=2, color='DarkSlateGrey')),
    name=LegendItemFig2[2],
    hovertemplate='%{x|%b %Y}<br>Latest Nowcast: %{y:.3f} Bln €<extra></extra>'
))

# Update layout
fig2.update_layout(
    yaxis=dict(title="Nominal GDP (billions of euros)", range=[350, 530], tickvals=list(range(400, 601, 50))),
    xaxis=dict(tickformat='%Y', tickangle=45, dtick="M12"),
    font=dict(family="Times New Roman", size=18),
    width=width0,
    height=height0,
    legend=dict(
        orientation="h",
        yanchor="bottom", y=1.02,
        xanchor="center", x=0.5
    )
)

fig2.show()
fig2_path = os.path.join(base_path, "HISTORY/FIGURES/", f"json_{vintage}", "GDP_Level_Ncast.json")
fig2.write_json(fig2_path)

In [ ]:
# ===================================================================
# 4. FIGURE: Distribution of Nowcasts
# ===================================================================

print("--- Generating Figure 3: Distribution of Final Nowcasts ---")

# Extract the last row of nowcasts and drop any missing values
last_nowcasts = y_predicted.iloc[-1].dropna()
nowcast_values = last_nowcasts.values

# Store the key model values to be highlighted
bg_value = last_nowcasts.get('BG_4')
bg_k1_value = last_nowcasts.get('BG_K1_4')
median_value = last_nowcasts.get('Median')

# Perform Kernel Density Estimation
kde = gaussian_kde(nowcast_values)
# Create a range of x-values for a smooth plot
x_range = np.linspace(nowcast_values.min() - 0.005, nowcast_values.max() + 0.005, 500)
kde_y = kde(x_range)

# Plot with Plotly
fig3 = go.Figure()

# Add the shaded KDE plot
fig3.add_trace(go.Scatter(
    x=x_range,
    y=kde_y,
    mode='lines',
    fill='tozeroy',
    fillcolor='rgba(0, 114, 189, 0.2)', # Light blue fill
    line=dict(color='#0072BD', width=2),
    name='Density (KDE)'
))

# --- MODIFICATION: Plot key nowcasts as points on the density curve ---
marker_style = dict(size=15, line=dict(width=2, color='DarkSlateGrey'))

if bg_value is not None:
    fig3.add_trace(go.Scatter(
        x=[bg_value], y=kde(bg_value),
        mode='markers',
        marker=dict(color='Teal', **marker_style),
        name=f"Full: {bg_value:.3%}"
    ))

if bg_k1_value is not None:
    fig3.add_trace(go.Scatter(
        x=[bg_k1_value], y=kde(bg_k1_value),
        mode='markers',
        marker=dict(color='Teal', **marker_style),
        name=f"Skilled: {bg_k1_value:.3%}"
    ))

if median_value is not None:
    fig3.add_trace(go.Scatter(
        x=[median_value], y=kde(median_value),
        mode='markers',
        marker=dict(color='ForestGreen', **marker_style),
        name=f"Median: {median_value:.3%}"
    ))
    
# Update layout
fig3.update_layout(
    xaxis_title="Quarterly Nominal GDP Growth",
    yaxis_title="Density",
    xaxis_tickformat=".2%",
    font=dict(family="Times New Roman", size=18),
    width=width0,
    height=height0,
    legend=dict(
        orientation="h",
        yanchor="bottom", y=1.02,
        xanchor="center", x=0.5
    )
)

fig3.show()
fig3_path = os.path.join(base_path, "HISTORY/FIGURES/", f"json_{vintage}", "GDP_KDE_Ncast.json")
fig3.write_json(fig3_path)

In [ ]:
# ===================================================================
# 5. SUMMARY TABLES
# ===================================================================

print("\n--- Generating Summary Tables ---")

# --- Table 1: Monthly (end-of-month) update measures ---
NowTab1_data = {
    "QoQ % Change (back)": 100 * y_growth['BG_K1_4'].iloc[-2],
    "QoQ % Change (now)": 100 * y_growth['BG_K1_4'].iloc[-1],
    f"{int(a*100)}% Pr. (now)": 100 * y_growth['BG_K1_4_Low'].iloc[-1],
    f"{int((1-a)*100)}% Pr. (now)": 100 * y_growth['BG_K1_4_Upp'].iloc[-1],
}
NowTab1 = pd.DataFrame(NowTab1_data, index=[last_date.strftime('%Y-%m-%d')])
NowTab1 = NowTab1.round(2)

print("\nTable 1: Monthly (end-of-month) update measures.")
print(NowTab1.to_string())

# --- Table 2: Quarterly (end-of-quarter) summary measures ---
# YoY Growth: (Nowcast(t) - Actual(t-12)) / Actual(t-12)
YoYPercGrowth = 100 * (y_level['BG_K1_4'].iloc[-1] - y_level['Level'].iloc[-13]) / y_level['Level'].iloc[-13]

# Carry-over Growth: (Nowcast(t) - Last_Actual_of_Prev_Year) / Last_Actual_of_Prev_Year
prev_year_filter = y_level.index.year == (last_date.year - 1)
last_level_prev_year = y_level.loc[prev_year_filter, 'Level'].iloc[-1]
CarryGrowth = 100 * (y_level['BG_K1_4'].iloc[-1] - last_level_prev_year) / last_level_prev_year

NowTab2_data = {
    "QoQ % Change": 100 * y_growth['BG_K1_4'].iloc[-1],
    "YoY % Change": YoYPercGrowth,
    "Carry Over": CarryGrowth,
    "Level (Bln €)": y_level['BG_K1_4'].iloc[-1]
}
NowTab2 = pd.DataFrame(NowTab2_data, index=[f"Q{last_date.quarter}-{last_date.year}"])
NowTab2 = NowTab2.round(2)

print("\nTable 2: Quarterly (end-of-quarter) summary measures.")
print(NowTab2.to_string())

# --- Save tables to CSV files ---
try:
    rt_tables_path = os.path.join(base_path, "HISTORY/RT_TABLES/")
    os.makedirs(rt_tables_path, exist_ok=True)
    NowTab1.to_csv(os.path.join(rt_tables_path, f"growth_fcast_{vintage}.csv"))
    NowTab2.to_csv(os.path.join(rt_tables_path, f"level_fcast_{vintage}.csv"))
    print("\nSummary tables saved successfully.")
except Exception as e:
    print(f"\nError saving tables: {e}")